# 03 · Train / test split

Validation is the two months after training ends, so I test the same way: train on the past, test on the next two months.

| Fold | Train on | Test on |
|---|---|---|
| 1 | Jan–Apr | May–Jun |
| 2 | Jan–Jun | Jul–Aug |
| 3 | Jan–Aug | Sep–Oct (closest to the real Nov–Dec task) |

Cleaning and outlier flags are refit inside each fold on its training rows only, so nothing from the test period leaks in.

## The three folds

In [1]:
import sys; sys.path.append("../src")
import pandas as pd
from splits import make_folds, prepare_fold, unseen_report
from metrics import evaluate

raw = pd.read_csv("../data/train_test.csv")

rows = []
for k, tr_idx, te_idx in make_folds(raw):
    train, test, _, _ = prepare_fold(raw, tr_idx, te_idx)
    rows.append({
        "fold": k,
        "train_rows": len(train), "test_rows": len(test),
        "train_end": train["date"].max().date(),
        "test_start": test["date"].min().date(),
        "test_end": test["date"].max().date(),
        "test_outlier_rate": round(test["is_outlier"].mean(), 4),
        **{key: round(v, 4) for key, v in unseen_report(train, test).items()},
    })
pd.DataFrame(rows)

,fold,train_rows,test_rows,train_end,test_start,test_end,test_outlier_rate,unseen_pickup,unseen_delivery,unseen_lane
0,1,19110,9696,2025-04-30,2025-05-01,2025-06-30,0.0129,0.0,0.0,0.0183
1,2,28806,9671,2025-06-30,2025-07-01,2025-08-31,0.0138,0.0,0.0,0.0054
2,3,38477,9523,2025-08-31,2025-09-01,2025-10-31,0.0151,0.0,0.0,0.0022


## First baseline: distance × one median rate per mile
Gives a floor that every real model has to beat.

In [2]:
# Predict: distance x median rate-per-mile of the fold's training rows
results = []
for k, tr_idx, te_idx in make_folds(raw):
    train, test, _, _ = prepare_fold(raw, tr_idx, te_idx)
    fit_rows = train[~train["is_outlier"]]           # DECISION: drop flagged outliers when fitting
    rpm = (fit_rows["posted_rate"] / fit_rows["distance"]).median()
    pred = test["distance"] * rpm
    results.append({"fold": k, **evaluate(test["posted_rate"], pred, test["is_outlier"])})

res = pd.DataFrame(results).set_index("fold")
display(res.round(2))
print(res.mean().round(2))

,MAE,RMSE,MAPE%,MAE_clean,RMSE_clean,MAPE%_clean
fold,,,,,,
1,241.78,671.84,11.89,189.83,244.87,9.82
2,249.02,674.72,11.16,199.15,289.88,9.03
3,256.95,684.25,11.63,203.69,295.92,9.27


MAE            249.25
RMSE           676.94
MAPE%           11.56
MAE_clean      197.56
RMSE_clean     276.89
MAPE%_clean      9.37
dtype: float64


## Unseen-city stress test
Validation has 8 cities the model never saw. To mimic that, I hold 8 random cities out of training and score only the test loads that touch them.

In [3]:
import numpy as np

rng = np.random.default_rng(42)
k, tr_idx, te_idx = list(make_folds(raw))[-1]          # fold 3
train, test, _, _ = prepare_fold(raw, tr_idx, te_idx)

cities = sorted(set(train["pickup"]) | set(train["delivery"]))
held = set(rng.choice(cities, size=8, replace=False))   # mimic the 8 unseen cities
print("held-out cities:", held)

involves = lambda d: d["pickup"].isin(held) | d["delivery"].isin(held)
train_hs = train[~involves(train)]
test_hs = test[involves(test)]
print("train rows kept:", len(train_hs), "| hard test rows:", len(test_hs))

held-out cities: {np.str_('Toledo'), np.str_('San Antonio'), np.str_('Hartford'), np.str_('Tulsa'), np.str_('Memphis'), np.str_('Washington'), np.str_('Bakersfield'), np.str_('New York')}
train rows kept: 28431 | hard test rows: 2516
